# CASMI 完整流程：正式提交入口
全部分支强制执行：谱图清洗→中性质量→实验谱库检索＋PubChem/COCONUT/LOTUS质量召回＋分子式注释＋de novo→神经评分＋物理断键评分→多谱融合／RRF→官方结构去重→25个SMILES。缺资产或权重会报错；不会生成占位提交。

在 Kaggle 设置中关闭 Internet，选择 GPU；评分时会读取替换后的 test.parquet。

In [ ]:
# The scoring rerun has no Internet. Mount a wheel dataset prepared for this Python version.
import sys, subprocess, importlib.metadata
from pathlib import Path
WHEEL_DIRS = [Path('/kaggle/input/casmi26-rdkit-2026-03-3-wheel'), Path('/kaggle/input/casmi-offline-wheels')]
try:
    installed = importlib.metadata.version('rdkit')
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed not in ('2026.3.3', '2026.03.3'):
    wheel_files = []
    for directory in WHEEL_DIRS:
        if directory.is_dir():
            wheel_files.extend(directory.rglob('rdkit*.whl'))
    if not wheel_files and Path('/kaggle/input').is_dir():
        wheel_files = list(Path('/kaggle/input').rglob('rdkit*.whl'))
    if not wheel_files:
        raise FileNotFoundError(f'Mount pinned RDKit2026.03.3 wheel dataset; input roots={list(Path('/kaggle/input').iterdir()) if Path('/kaggle/input').is_dir() else []}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', str(wheel_files[0])], check=True)
import rdkit, torch, pandas, pyarrow
assert rdkit.__version__ == '2026.03.3', rdkit.__version__
print('RDKit', rdkit.__version__, 'PyTorch', torch.__version__, 'CUDA', torch.cuda.is_available())


In [ ]:
# Exact source bundle generated from the local project; no remote code downloads.
import base64, io, zipfile, sys
from pathlib import Path
CODE_ROOT = Path('/kaggle/working/casmi_full_code')
CODE_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE))) as archive:
    for name in archive.namelist():
        target = (CODE_ROOT/name).resolve()
        if not target.is_relative_to(CODE_ROOT.resolve()):
            raise ValueError('Unexpected archive path')
    archive.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print('Loaded', len(list((CODE_ROOT/'casmi').glob('*.py'))), 'project modules')


In [ ]:
import json, time, shutil, subprocess
from casmi.config import load_config
from casmi.preparation import prepare_catalog
from casmi.pipeline import FullPipeline

# Kaggle-only bootstrap: public inputs are mounted by kernel metadata.
# This creates the exact catalog/checkpoint formats consumed by FullPipeline.
WORK = Path('/kaggle/working/casmi_assets')
WORK.mkdir(parents=True, exist_ok=True)
CATALOG = WORK / 'catalog.sqlite'
PAIRED = WORK / 'paired'
MODEL_DIR = WORK / 'models'
PREPROCESSING = {'intensity_floor':0.001, 'max_peaks':128, 'top_per_window':6}
def find_input(filename, required_text=()):
    roots = [Path('/kaggle/input'), Path('/kaggle/input/datasets'), Path('/kaggle/input/competitions')]
    matches = []
    for root in roots:
        if root.is_dir():
            matches.extend(p for p in root.rglob(filename) if p.is_file())
    matches = [p for p in matches if all(t.lower() in str(p).lower() for t in required_text)] or matches
    if not matches:
        raise FileNotFoundError(f'Missing mounted public asset {filename}; input roots={list(Path('/kaggle/input').rglob('*'))[:30]}')
    return sorted(set(matches), key=str)[0]
competition_train = find_input('train.parquet', ('enveda',))
competition_test = find_input('test.parquet', ('enveda',))
competition_sample = find_input('sample_submission.csv', ('enveda',))
external = find_input('external_candidates.parquet', ('pubchem',))
lotus = find_input('candidate_db_coconut_lotus.parquet', ())

# Build a compact, test-mass-focused asset set. The public files are large;
# indexing every molecule and every training spectrum is unnecessary for inference.
import pandas as pd, numpy as np
import pyarrow.parquet as pq
from casmi.spectra import spectrum_from_row
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
prep = WORK / 'prep_sources'; prep.mkdir(exist_ok=True)
test_rows = pd.read_parquet(competition_test)
target_masses = []
for row in test_rows.to_dict('records'):
    try:
        target_masses.append(float(spectrum_from_row(row, PREPROCESSING).neutral_mass))
    except Exception:
        pass
target_masses = np.array(sorted(set(round(x, 6) for x in target_masses if np.isfinite(x))), dtype='float64')
if len(target_masses) < 10:
    raise RuntimeError(f'Could not derive test neutral masses; got {len(target_masses)}')
def near_target(values, ppm=150.0):
    values = np.asarray(values, dtype='float64')
    pos = np.searchsorted(target_masses, values)
    pos0 = np.clip(pos, 0, len(target_masses)-1)
    pos1 = np.clip(pos-1, 0, len(target_masses)-1)
    return (np.abs(values-target_masses[pos0]) <= target_masses[pos0]*ppm*1e-6) | (np.abs(values-target_masses[pos1]) <= target_masses[pos1]*ppm*1e-6)
def nearest_per_target(frame, mass_col='exact_mass', per_target=100, ppm=50.0):
    frame = frame.dropna(subset=[mass_col]).drop_duplicates('smiles').copy()
    values = frame[mass_col].to_numpy(dtype='float64')
    order = np.argsort(values, kind='stable')
    values = values[order]
    chosen = set()
    for target in target_masses:
        tol = target * ppm * 1e-6
        lo = np.searchsorted(values, target - tol, side='left')
        hi = np.searchsorted(values, target + tol, side='right')
        if hi > lo:
            local = np.arange(lo, hi)
            local = local[np.argsort(np.abs(values[local] - target), kind='stable')[:per_target]]
            chosen.update(order[local].tolist())
    if not chosen:
        return frame.iloc[:0]
    return frame.iloc[sorted(chosen)].reset_index(drop=True)
external_df = pd.read_parquet(external)
pubchem_mask = external_df['sources'].astype(str).str.contains('pubchemlite', case=False, na=False)
coconut_mask = external_df['in_coconut'].fillna(False).astype(bool)
pubchem_df = nearest_per_target(external_df.loc[pubchem_mask], per_target=100)[['smiles']]
coconut_df = nearest_per_target(external_df.loc[coconut_mask], per_target=100)[['smiles']]
lotus_df = pd.read_parquet(lotus)
lotus_mask = lotus_df['in_lotus'].fillna(False).astype(bool)
lotus_df = nearest_per_target(lotus_df.loc[lotus_mask], per_target=100)[['smiles']]
for label, frame in [('pubchem', pubchem_df), ('coconut', coconut_df), ('lotus', lotus_df)]:
    if len(frame) < 25:
        raise RuntimeError(f'{label} mass-filtered asset has only {len(frame)} structures')
    print(label, 'structures=', len(frame))
pubchem_path = prep / 'pubchem.parquet'; coconut_path = prep / 'coconut.parquet'; lotus_path = prep / 'lotus.parquet'
pubchem_df.to_parquet(pubchem_path, index=False); coconut_df.to_parquet(coconut_path, index=False); lotus_df.to_parquet(lotus_path, index=False)
# Read only an initial bounded training slice for the experimental-library branch.
# It is enough to make the branch real while keeping the submit kernel short.
train_path = prep / 'competition_subset.parquet'
if not train_path.exists():
    parquet_file = pq.ParquetFile(str(competition_train))
    collected = []
    # These list columns are nested in the competition Parquet schema; checking
    # schema.names would incorrectly discard them, so request the known schema directly.
    columns = ['normalized_smiles','molecule_id','spectrum_id','ms2_mzs','ms2_normalized_intensities','precursor_mz','adduct','ionization_mode','collision_energy_ev','instrument_type','base_peak_intensity','precursor_error_ppm']
    try:
        batches = parquet_file.iter_batches(batch_size=4096, columns=columns)
    except Exception as exc:
        raise RuntimeError(f'Competition train schema is incompatible with expected peak columns: {parquet_file.schema_arrow.names}') from exc
    for batch in batches:
        collected.extend(batch.to_pylist())
        if len(collected) >= 30000:
            break
    if len(collected) < 100:
        raise RuntimeError(f'Competition subset unexpectedly small: {len(collected)}')
    pd.DataFrame(collected[:30000]).to_parquet(train_path, index=False)
manifest = {
  'max_precursor_error_ppm': 20, 'preprocessing': PREPROCESSING, 'resources': [
    {'source':'competition','kind':'spectra','path':str(train_path),'source_url':'https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data','license':'CC-BY-NC-4.0','supervised':True},
    {'source':'pubchem','kind':'structures','path':str(pubchem_path),'source_url':'https://pubchem.ncbi.nlm.nih.gov/','license':'CC-BY-4.0'},
    {'source':'coconut','kind':'structures','path':str(coconut_path),'source_url':'https://coconut.naturalproducts.net/','license':'CC-BY-4.0'},
    {'source':'lotus','kind':'structures','path':str(lotus_path),'source_url':'https://lotus.naturalproducts.net/','license':'Apache-2.0'}
  ]
}
manifest_path = WORK / 'manifest.json'; manifest_path.write_text(json.dumps(manifest, indent=2))
if not CATALOG.exists():
    report = prepare_catalog(manifest_path, CATALOG, PAIRED)
    print('Prepared compact catalog', json.dumps({k:report.get(k) for k in ('n_structures','n_spectra','structure_sources')}, ensure_ascii=False))

preprocess_path = WORK / 'preprocessing.json'; preprocess_path.write_text(json.dumps(PREPROCESSING))
if not (MODEL_DIR / 'neural.pt').is_file() or not (MODEL_DIR / 'denovo.pt').is_file():
    MODEL_DIR.mkdir(exist_ok=True)
    subprocess.run([sys.executable, str(CODE_ROOT/'scripts/train_models.py'),
        '--train', str(PAIRED), '--output-dir', str(MODEL_DIR), '--stage', 'both',
        '--max-rows', '10000', '--steps', '100', '--batch-size', '64',
        '--width', '48', '--heads', '4', '--layers', '1', '--decoder-layers', '1',
        '--fp-bits', '512', '--max-length', '96', '--preprocessing-json', str(preprocess_path),
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu'], check=True)

CONFIG = load_config(CODE_ROOT/'configs/inference.example.json')
CONFIG['assets'] = {'structures':str(CATALOG),'spectra':str(CATALOG),'neural':str(MODEL_DIR/'neural.pt'),'denovo':str(MODEL_DIR/'denovo.pt'),'manifest':str(CATALOG.with_suffix('.audit.json'))}
CONFIG['input'] = {'test':str(competition_test),'sample':str(competition_sample)}
CONFIG['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
CONFIG['output_dir'] = '/kaggle/working/full_run'
CONFIG['require_25'] = True
CONFIG['mass_ppm'] = 50.0
CONFIG['formula']['ppm'] = 50.0
CONFIG['structures_per_source'] = 300
CONFIG['library_hits_per_spectrum'] = 20
CONFIG['physical_shortlist'] = 25
CONFIG['physical']['max_bond_cuts'] = 1
CONFIG['physical']['max_cutsets'] = 500
CONFIG['physical']['max_fragments'] = 64
CONFIG['physical']['max_bonds'] = 80
CONFIG['generation']['max_candidates'] = 16
CONFIG['max_seconds'] = 8.5 * 3600
CONFIG['require_rdkit_version'] = '2026.03.3'
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


In [ ]:
started = time.monotonic()
pipeline = FullPipeline(CONFIG)
submission = pipeline.run()
expected = {'library_retrieval','pubchem_retrieval','coconut_retrieval','lotus_retrieval','formula_annotation','denovo_generation','neural_scoring','physical_scoring','multi_spectrum_fusion','identity_deduplication'}
counts = pipeline.audit['stage_counts']
assert all(counts.get(stage) == len(submission) for stage in expected), counts
assert (submission.smiles.str.count(';') == 24).all()
shutil.copy2(Path(CONFIG['output_dir'])/'submission.csv', '/kaggle/working/submission.csv')
print('All branches ran for', len(submission), 'molecules; seconds=', round(time.monotonic()-started,2))
submission.head()


In [ ]:
report = json.loads((Path(CONFIG['output_dir'])/'run_report.json').read_text())
assert report['success'] is True
print('Stage counts:', report['stage_counts'])
print('Read candidate_scores.parquet and run_report.json for per-source evidence and generation failures/truncations.')
print('Ready output: /kaggle/working/submission.csv')
